In [ ]:
!uv pip install "git+https://github.com/Armandpl/ai_image_models.git

In [ ]:
from pathlib import Path

import kagglehub
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torchvision.utils import make_grid

In [ ]:
data_dir = Path(kagglehub.dataset_download("ebrahimelgazar/pixel-art"))
print("downloaded to:", data_dir)

In [ ]:
!ls {data_dir}

In [ ]:
sprites = np.load(data_dir / "sprites.npy", allow_pickle=False)
print("sprites.npy", sprites.shape, sprites.dtype, "range", (sprites.min(), sprites.max()))

sprites_labels = np.load(data_dir / "sprites_labels.npy", allow_pickle=False)
print("\nsprites_labels.npy", sprites_labels.shape, sprites_labels.dtype, "range", (sprites_labels.min(), sprites_labels.max()))

In [ ]:
class PixelArtDataset(Dataset):
      def __init__(self, data_dir):
          # (N, H, W, 3) uint8 -> (N, H, W, 3) float32 in [0, 1]
          images = np.load(data_dir / "sprites.npy", allow_pickle=False)
          labels = np.load(data_dir / "sprites_labels.npy", allow_pickle=False)
          self.x = torch.from_numpy(images).float() / 255.0
          self.y = torch.from_numpy(labels).float()  # one-hot (N, 5)

      def __len__(self) -> int:
          return len(self.x)
  
      def __getitem__(self, i: int):
          return self.x[i], self.y[i]

In [ ]:
ds = PixelArtDataset(data_dir)
n_per = 8
cls = ds.y.argmax(1)
classes = cls.unique().tolist()
fig, axes = plt.subplots(len(classes), n_per, figsize=(n_per, len(classes)))
for row, c in enumerate(classes):
  idx = (cls == c).nonzero().squeeze(1)[:n_per]
  for ax, i in zip(axes[row], idx):
      img, label = ds[i]
      ax.imshow(img.numpy()); ax.axis("off")
  axes[row, 0].set_ylabel(f"class {c}", rotation=0, labelpad=20, va="center")
plt.tight_layout(); plt.show()

In [ ]:
from ai_image_models.models import FlowMLP
from ai_image_models.learner import Learner

model = FlowMLP(img_shape=(16, 16, 3))
learner = Learner(model)
learner.learn(ds, epochs=50)

In [ ]:
imgs = learner.generate(n=64)
grid = make_grid(imgs.permute(0, 3, 1, 2), nrow=8)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).numpy()); plt.axis("off"); plt.show()

In [ ]:
from PIL import Image

traj = learner.generate(n=64, trajectory=True) # (steps, n, 16, 16, 3)
frames = []
for step in traj[::5]:
  grid = make_grid(step.permute(0, 3, 1, 2), nrow=8)
  arr = (grid.permute(1, 2, 0).numpy() * 255).astype("uint8")
  frames.append(Image.fromarray(arr))

frames[0].save("sprites.gif", save_all=True, append_images=frames[1:], duration=10, loop=0)